# Imports

In [1]:
import json
import time
from dataclasses import dataclass, asdict
from pathlib import Path

from pynput import mouse, keyboard

RECORDINGS_DIR = Path("recordings")
RECORDINGS_DIR.mkdir(exist_ok=True)

## Recorder

In [2]:
def record(stop_key=keyboard.Key.esc):
    events = []
    start = time.perf_counter()

    def now():
        return time.perf_counter() - start

    def on_click(x, y, button, pressed):
        events.append({
            "t": now(),
            "type": "click",
            "x": x,
            "y": y,
            "button": button.name,
            "pressed": pressed,
        })

    def on_scroll(x, y, dx, dy):
        events.append({
            "t": now(),
            "type": "scroll",
            "x": x,
            "y": y,
            "dx": dx,
            "dy": dy,
        })

    def on_press(key):
        if key == stop_key:
            return False  # stops the keyboard listener
        events.append({
            "t": now(),
            "type": "key",
            "key": _key_to_str(key),
            "pressed": True,
        })

    def on_release(key):
        events.append({
            "t": now(),
            "type": "key",
            "key": _key_to_str(key),
            "pressed": False,
        })

    mouse_listener = mouse.Listener(on_click=on_click, on_scroll=on_scroll)
    keyboard_listener = keyboard.Listener(on_press=on_press, on_release=on_release)

    mouse_listener.start()
    keyboard_listener.start()
    print("Recording... press ESC to stop.")

    keyboard_listener.join()  # blocks until on_press returns False
    mouse_listener.stop()

    print(f"Recorded {len(events)} events over {events[-1]['t']:.2f}s" if events else "No events recorded.")
    return events


def _key_to_str(key):
    """Serialize a pynput key into a string that _str_to_key can parse back."""
    if isinstance(key, keyboard.KeyCode):
        return f"char:{key.char}"
    return f"special:{key.name}"


def _str_to_key(s):
    kind, value = s.split(":", 1)
    if kind == "char":
        return keyboard.KeyCode.from_char(value)
    return getattr(keyboard.Key, value)

In [5]:
# Run this cell, then switch to whatever window you want to record clicks/keys in.
events = record()

Recording... press ESC to stop.
Recorded 2 events over 1.36s


In [6]:
# Save the recording to a JSON file for later playback.
recording_name = "clear_all"  # change this per recording

out_path = RECORDINGS_DIR / f"{recording_name}.json"
out_path.write_text(json.dumps(events, indent=2))
print(f"Saved to {out_path}")

Saved to recordings\clear_all.json
